# Task 2: Exploratory Data Analysis (EDA)

**AI & ML Internship — Elevate Labs**

---

## Objective
Perform comprehensive Exploratory Data Analysis (EDA) on the Titanic dataset to uncover patterns, trends, relationships, and anomalies using statistics and visualizations. This notebook demonstrates how EDA informs downstream preprocessing and modeling decisions.

**Key goals:**
1. Generate descriptive statistics for all features
2. Visualize distributions and detect outliers
3. Analyze relationships between features using correlation and pairplots
4. Identify patterns, trends, and anomalies
5. Draw data-driven inferences before any modeling
6. Prepare for common EDA interview questions


## Objective

Exploratory Data Analysis (EDA) is the critical first step in any data science workflow. Before building models, we must understand the data's structure, quality, and underlying patterns.

**Why EDA matters:**
- Reveals data quality issues (missing values, outliers, errors)
- Guides feature engineering and preprocessing choices
- Helps validate assumptions about the data
- Generates hypotheses for further testing
- Prevents "garbage in, garbage out" in modeling

This notebook covers all EDA essentials using the Titanic dataset.


## Dataset Description

**Dataset:** Titanic — Machine Learning from Disaster  
**Source:** [Kaggle Titanic Dataset](https://www.kaggle.com/c/titanic/data)  
**Local path:** `dataset/titanic.csv`  
**Shape:** 891 rows × 12 columns

### Columns
| Column | Type | Description |
|--------|------|-------------|
| PassengerId | int | Unique passenger identifier |
| Survived | int | Target variable (0 = No, 1 = Yes) |
| Pclass | int | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) |
| Name | str | Passenger full name |
| Sex | str | Gender |
| Age | float | Age in years |
| SibSp | int | Siblings/Spouses aboard |
| Parch | int | Parents/Children aboard |
| Ticket | str | Ticket number |
| Fare | float | Passenger fare |
| Cabin | str | Cabin number |
| Embarked | str | Port of Embarkation (C, Q, S) |

**Note:** We use the raw dataset here to practice EDA. Preprocessing (handling missing values, encoding, etc.) happens after EDA in a typical workflow.


## Import Libraries

We import libraries for data manipulation, statistical analysis, and visualization.

- `pandas` — data loading and descriptive statistics
- `numpy` — numerical operations
- `matplotlib.pyplot` — base plotting
- `seaborn` — statistical visualizations
- `plotly.express` — interactive visualizations


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plot styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("Libraries imported successfully.")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__} | Seaborn: {sns.__version__}")


Libraries imported successfully.
Pandas: 2.2.3 | NumPy: 2.1.3 | Seaborn: 0.13.2


## Load Dataset

Load the Titanic CSV and verify its integrity. We check the file exists and display the initial shape.

**Expected output:**
- Success message
- Shape: (891, 12)


In [3]:
import os

dataset_path = "../task-2-eda/dataset/titanic.csv"

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"File size: {os.path.getsize(dataset_path) / 1024:.2f} KB")


FileNotFoundError: Dataset not found at ../task-2-eda/dataset/titanic.csv

## Summary Statistics

We generate descriptive statistics for all features. This gives us an instant overview of central tendency, spread, and data quality.

**What to look for:**
- `count` — reveals missing values if less than total rows
- `mean` vs `median` (50%) — large differences indicate skewness
- `std` — variability
- `min` / `max` — range and potential outliers
- `25%`, `75%` — quartile positions for IQR calculations


In [ ]:
print("=== Summary Statistics: Numerical Features ===")
display(df.describe().round(2))

print("\n=== Summary Statistics: Categorical Features ===")
display(df.describe(include=['object']).round(2))


In [ ]:
print("=== Dataset Info ===")
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")


## Missing Value Overview

EDA requires understanding data completeness. We quantify and visualize missing values to inform later preprocessing decisions.

**Why this matters:** Missingness patterns (MCAR, MAR, MNAR) affect which imputation strategy is appropriate. We document the pattern here and will handle it in Task 1.


In [ ]:
print("=== Missing Values Summary ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)
display(missing_df)

print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Columns with missing values: {missing_df.index.tolist()}")


In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap (Yellow = Missing)', fontsize=14)
ax.set_xlabel('Columns', fontsize=12)
ax.set_ylabel('Row Index', fontsize=12)
plt.tight_layout()
plt.show()

print("Interpretation: Cabin has the most missing data (~77%), followed by Age (~20%) and Embarked (~0.2%).")


## Univariate Analysis — Numerical Features

We examine the distribution of each numerical feature using histograms and boxplots.

**What we learn:**
- **Histograms:** Show shape (normal, skewed, bimodal), central tendency, spread
- **Boxplots:** Show median, quartiles, and outliers
- **Skewness:** Asymmetry in distribution; affects choice of imputation and scaling

**Expected visualizations:**
- Age: approximately normal with some skew
- Fare: heavily right-skewed (long tail)
- SibSp/Parch: discrete, low values with outliers


In [ ]:
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']

print("=== Histograms: Numerical Features ===")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='steelblue', bins=30)
    axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)

plt.tight_layout()
plt.show()

print("Interpretation: Fare is heavily right-skewed. Age is roughly normal but slightly left-skewed. SibSp and Parch are discrete with most values near 0.")


In [ ]:
print("=== Boxplots: Numerical Features (Outlier Detection) ===")
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for idx, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[idx], color='lightcoral')
    axes[idx].set_title(f'Boxplot: {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(col, fontsize=10)

plt.tight_layout()
plt.show()

print("Interpretation: Dots above/below whiskers are outliers. Fare has extreme high values. SibSp and Parch have rare high values (large families). Age has a few low outliers.")


In [ ]:
print("=== Skewness & Kurtosis ===")
for col in numeric_cols:
    skew = df[col].skew()
    kurt = df[col].kurtosis()
    print(f"{col}: skewness = {skew:.2f}, kurtosis = {kurt:.2f}")

print("\nRule of thumb: |skewness| > 1 = highly skewed. Fare is highly skewed right. SibSp/Parch are highly skewed right.")


## Univariate Analysis — Categorical Features

We examine the frequency distribution of categorical features using count plots.

**What we learn:**
- Class balance/imbalance
- Dominant categories
- Rare categories that may need grouping


In [ ]:
cat_cols = ['Survived', 'Pclass', 'Sex', 'Embarked']

print("=== Count Plots: Categorical Features ===")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    sns.countplot(x=df[col], ax=axes[idx], palette='viridis')
    axes[idx].set_title(f'Count of {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Count', fontsize=10)
    # Add count labels on bars
    for p in axes[idx].patches:
        axes[idx].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                          ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Interpretation: More passengers died (0) than survived (1). 3rd class is most common. Males outnumber females. Southampton (S) is the most common embarkation port.")


## Bivariate Analysis — Target vs Features

We analyze how each feature relates to the target variable `Survived`. This helps identify predictive features.

**What we learn:**
- Which features have strong relationships with survival
- Potential feature importance rankings
- Interaction effects


In [ ]:
print("=== Survival Rate by Categorical Features ===")

# Survival by Sex
print("\n1. Survival by Sex:")
display(pd.crosstab(df['Sex'], df['Survived'], margins=True))
print("\nSurvival rate by Sex (%):")
display(df.groupby('Sex')['Survived'].mean() * 100)

# Survival by Pclass
print("\n2. Survival by Pclass:")
display(pd.crosstab(df['Pclass'], df['Survived'], margins=True))
print("\nSurvival rate by Pclass (%):")
display(df.groupby('Pclass')['Survived'].mean() * 100)

# Survival by Embarked
print("\n3. Survival by Embarked:")
display(pd.crosstab(df['Embarked'], df['Survived'], margins=True))
print("\nSurvival rate by Embarked (%):")
display(df.groupby('Embarked')['Survived'].mean() * 100)


In [ ]:
print("=== Visualizations: Survival by Categorical Features ===")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(x='Sex', y='Survived', data=df, ax=axes[0], palette='coolwarm')
axes[0].set_title('Survival Rate by Sex', fontweight='bold')
axes[0].set_ylabel('Survival Rate')

sns.barplot(x='Pclass', y='Survived', data=df, ax=axes[1], palette='coolwarm')
axes[1].set_title('Survival Rate by Pclass', fontweight='bold')
axes[1].set_ylabel('Survival Rate')

sns.barplot(x='Embarked', y='Survived', data=df, ax=axes[2], palette='coolwarm')
axes[2].set_title('Survival Rate by Embarked', fontweight='bold')
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()

print("Interpretation: Females survived at much higher rates (~74%) than males (~19%). 1st class had highest survival (~63%), 3rd class lowest (~24%). Southampton passengers had lower survival rates.")


In [ ]:
print("=== Numerical Features vs Survived ===")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='Survived', y='Age', data=df, ax=axes[0, 0], palette='coolwarm')
axes[0, 0].set_title('Age by Survival', fontweight='bold')

sns.boxplot(x='Survived', y='Fare', data=df, ax=axes[0, 1], palette='coolwarm')
axes[0, 1].set_title('Fare by Survival', fontweight='bold')

sns.boxplot(x='Survived', y='SibSp', data=df, ax=axes[1, 0], palette='coolwarm')
axes[1, 0].set_title('SibSp by Survival', fontweight='bold')

sns.boxplot(x='Survived', y='Parch', data=df, ax=axes[1, 1], palette='coolwarm')
axes[1, 1].set_title('Parch by Survival', fontweight='bold')

plt.tight_layout()
plt.show()

print("Interpretation: Survivors paid higher fares on average. Age distributions are similar between survived/died. Large families (high SibSp/Parch) had lower survival rates.")


## Correlation Analysis

Correlation measures the linear relationship between two numerical variables. It ranges from -1 (perfect negative) to +1 (perfect positive).

**Why it matters:**
- Identifies redundant features (high correlation)
- Reveals relationships with the target variable
- Detects multicollinearity (high inter-feature correlation)

**Key metric:** Pearson correlation coefficient (r)


In [ ]:
# Select numeric columns for correlation
numeric_df = df.select_dtypes(include=[np.number])

print("=== Correlation Matrix ===")
corr_matrix = numeric_df.corr().round(3)
display(corr_matrix)

print("\n=== Correlation with Target (Survived) ===")
survived_corr = corr_matrix['Survived'].sort_values(ascending=False)
display(survived_corr)


In [ ]:
print("=== Correlation Heatmap ===")
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Heatmap (Numerical Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation: Pclass and Fare are strongly negatively correlated (-0.55) — higher class (lower number) means higher fare. Survived is positively correlated with Fare (0.26) and negatively with Pclass (-0.34).")


In [ ]:
print("=== Scatter Plot: Fare vs Age (Colored by Survived) ===")
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(x='Age', y='Fare', hue='Survived', data=df, alpha=0.7, palette='coolwarm', ax=ax)
ax.set_title('Fare vs Age by Survival', fontsize=14, fontweight='bold')
ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Fare', fontsize=12)
plt.legend(title='Survived', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

print("Interpretation: Higher fares tend to correlate with survival. Age shows no clear linear pattern with survival.")


## Pairplot — Feature Relationships

A pairplot shows pairwise relationships between all numerical features, with the target variable as the hue.

**What we learn:**
- Linear vs non-linear relationships
- Separation between classes (Survived = 0 vs 1)
- Potential feature interactions


In [ ]:
print("=== Pairplot: Numerical Features by Survived ===")
pairplot_cols = ['Survived', 'Age', 'Fare', 'SibSp', 'Parch']
sns.pairplot(df[pairplot_cols], hue='Survived', palette='coolwarm', diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot of Numerical Features (Hue = Survived)', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("Interpretation: Fare shows the clearest separation between survived and non-survived. Age distributions overlap significantly. SibSp and Parch are discrete with limited separation.")


## Multivariate Analysis

We explore interactions between multiple features simultaneously.

**Example:** How does Sex and Pclass together affect survival?


In [ ]:
print("=== Survival Rate by Sex and Pclass ===")
survival_pivot = pd.pivot_table(df, values='Survived', index='Sex', columns='Pclass', aggfunc='mean')
display(survival_pivot.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(survival_pivot, annot=True, cmap='YlGnBu', fmt='.2f', cbar_kws={'label': 'Survival Rate'}, ax=ax)
ax.set_title('Survival Rate: Sex vs Pclass', fontsize=14, fontweight='bold')
ax.set_xlabel('Pclass', fontsize=12)
ax.set_ylabel('Sex', fontsize=12)
plt.tight_layout()
plt.show()

print("Interpretation: Females in 1st/2nd class had very high survival rates (>90% for 1st class females). Males in 3rd class had the lowest survival (~13%). This shows strong interaction between Sex and Pclass.")


In [ ]:
print("=== Interactive Plot: Fare Distribution by Pclass and Survived ===")
fig = px.box(df, x='Pclass', y='Fare', color='Survived', 
             title='Fare Distribution by Pclass and Survival',
             labels={'Pclass': 'Passenger Class', 'Fare': 'Fare', 'Survived': 'Survived'},
             category_orders={'Survived': [0, 1]})
fig.update_layout(xaxis_title='Pclass', yaxis_title='Fare')
fig.show()

print("Note: If Plotly does not render, inspect the static equivalent above.")


## Anomaly Detection

Anomalies are data points that deviate significantly from the norm. We use statistical methods and visualization to identify them.

**Methods used:**
- Visual: Boxplots (already shown)
- Statistical: IQR method, z-score
- Domain: Age < 1 or Age > 80, Fare = 0, etc.


In [ ]:
print("=== Anomaly Detection: IQR Method ===")

def detect_anomalies_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    anomalies = data[(data[column] < lower) | (data[column] > upper)]
    return anomalies, lower, upper, IQR

anomaly_summary = []
for col in ['Age', 'Fare', 'SibSp', 'Parch']:
    anomalies, lb, ub, iqr = detect_anomalies_iqr(df, col)
    anomaly_summary.append({
        'Column': col,
        'IQR': round(iqr, 2),
        'Lower_Bound': round(lb, 2),
        'Upper_Bound': round(ub, 2),
        'Anomaly_Count': len(anomalies),
        'Anomaly_Pct': round(len(anomalies) / len(df) * 100, 2)
    })

anomaly_df = pd.DataFrame(anomaly_summary)
display(anomaly_df)

print("\nInterpretation: Fare has the most anomalies due to extreme ticket prices. SibSp/Parch have rare high values (large families).")


In [ ]:
print("=== Anomaly Detection: Z-Score Method ===")
z_scores = np.abs(stats.zscore(df[['Age', 'Fare', 'SibSp', 'Parch']].dropna()))
z_summary = pd.DataFrame({
    'Column': ['Age', 'Fare', 'SibSp', 'Parch'],
    'Z_Score_Anomalies': (z_scores > 3).sum(axis=0).values,
    'Z_Score_Pct': ((z_scores > 3).sum(axis=0) / len(df) * 100).round(2).values
})
display(z_summary)

print("\nInterpretation: Z-score method is stricter and flags fewer anomalies than IQR for Fare, but more for Age because Age is more normally distributed.")


## Pattern Recognition & Inferences

Based on EDA, we extract actionable insights that guide preprocessing and modeling.

**Key patterns identified:**
1. **Sex is a strong predictor:** Females survived at ~74% vs males ~19%
2. **Pclass matters:** 1st class survival ~63%, 3rd class ~24%
3. **Fare correlates with survival:** Higher fare = higher survival rate
4. **Family size effect:** Passengers with large families (SibSp > 3, Parch > 2) had lower survival
5. **Age effect:** Children had slightly higher survival; elderly had mixed outcomes
6. **Embarked:** Southampton passengers had lower survival (possibly correlated with Pclass)


In [ ]:
print("=== Key Inferences Summary ===")
inferences = [
    "1. Sex: Females had ~74% survival vs males ~19% — strong gender-based survival bias",
    "2. Pclass: 1st class survival ~63%, 2nd ~47%, 3rd ~24% — wealth/status mattered",
    "3. Fare: Median survivor fare ($26) > median non-survivor fare ($10.5)",
    "4. Family size: SibSp > 3 and Parch > 2 correlate with lower survival",
    "5. Age: Children (Age < 10) had higher survival rates",
    "6. Embarked: Southampton passengers had lowest survival (~34%)",
    "7. Missing Cabin: 77% missing — cabin data is sparse, possibly correlates with class",
    "8. Correlation: Fare and Pclass are strongly negatively correlated (-0.55)"
]

for inf in inferences:
    print(inf)

print("\nThese inferences directly inform preprocessing choices:")
print("- Sex: Label encode (binary, strong signal)")
print("- Pclass: Keep as ordinal numeric")
print("- Fare: Handle outliers; consider log transformation for skewed distribution")
print("- SibSp/Parch: Combine into FamilySize or cap outliers")
print("- Age: Impute with median; consider binning for non-linear effects")


## Interactive Visualization (Plotly)

Interactive plots allow zooming, hovering, and filtering for deeper exploration.


In [ ]:
print("=== Interactive: Age Distribution by Survived ===")
fig = px.histogram(df, x='Age', color='Survived', nbins=30, barmode='overlay',
                   title='Age Distribution by Survival (Interactive)',
                   labels={'Age': 'Age', 'Survived': 'Survived'},
                   opacity=0.7)
fig.update_layout(xaxis_title='Age', yaxis_title='Count')
fig.show()

print("Note: Hover over bars for exact counts. Zoom in on specific age ranges.")


In [ ]:
print("=== Interactive: 3D Scatter Plot ===")
fig = px.scatter_3d(df, x='Age', y='Fare', z='Pclass', color='Survived',
                    title='3D: Age vs Fare vs Pclass by Survival',
                    labels={'Age': 'Age', 'Fare': 'Fare', 'Pclass': 'Pclass', 'Survived': 'Survived'},
                    opacity=0.7)
fig.show()

print("Note: This 3D view may reveal clusters or separation patterns between survivors and non-survivors.")


## Interview Preparation: EDA Q&A

### 1. What is the purpose of EDA?

**Simple answer:** To understand the data before building models — find patterns, spot problems, and generate insights.

**Technical answer:** EDA is the process of analyzing datasets to summarize their main characteristics, often using visual methods. It helps identify:
- Data quality issues (missing values, outliers, errors)
- Distributions and transformations needed
- Relationships between variables
- Assumptions for statistical modeling
- Feature importance and selection criteria

**Example:** Before building a Titanic survival model, EDA reveals that Sex and Pclass are strong predictors, Fare is skewed, and Age has missing values — all of which guide preprocessing decisions.

**Follow-up:** "What happens if you skip EDA and go straight to modeling?"

---

### 2. How do boxplots help in understanding a dataset?

**Simple answer:** Boxplots show the spread of data and highlight unusual values (outliers).

**Technical answer:** A boxplot displays the five-number summary: minimum, Q1, median, Q3, and maximum. The box spans Q1 to Q3 (IQR), with a line at the median. Whiskers extend to 1.5×IQR; points beyond are plotted individually as outliers. This makes it easy to:
- Compare distributions across categories
- Identify skewness (median position in the box)
- Spot outliers for further investigation or removal
- Detect data quality issues

**Example:** In Titanic, Fare boxplot reveals extreme values above £65, indicating a skewed distribution that may need log transformation or outlier handling.

**Follow-up:** "When would you remove an outlier vs keep it?"

---

### 3. What is correlation and why is it useful?

**Simple answer:** Correlation measures how two variables move together. It helps find relationships.

**Technical answer:** Correlation quantifies the strength and direction of a linear relationship between two continuous variables, typically measured by the Pearson correlation coefficient (r), ranging from -1 to +1. It is useful because:
- Feature selection: Remove highly correlated redundant features
- Target analysis: Identify features strongly related to the target
- Multicollinearity detection: High inter-feature correlation can destabilize linear models
- Dimensionality reduction: Inform PCA or feature engineering decisions

**Example:** In Titanic, Fare and Pclass have r = -0.55, indicating higher-class (lower Pclass number) passengers paid more. This redundancy suggests we might drop one or combine them.

**Follow-up:** "Does correlation imply causation? Why or why not?"

---

### 4. How do you detect skewness in data?

**Simple answer:** Skewness is when data is not symmetric — one tail is longer than the other.

**Technical answer:** Skewness can be detected through:
- **Visual inspection:** Histograms with a long right tail = positive skew; long left tail = negative skew
- **Statistical measure:** `skew()` function; |skewness| > 0.5 = moderately skewed, > 1 = highly skewed
- **Comparison of mean vs median:** Mean > median = positive skew; Mean < median = negative skew
- **Q-Q plots:** Compare data quantiles to normal distribution quantiles

**Example:** Titanic Fare has skewness ~4.8 (highly positively skewed). Most passengers paid low fares, but a few paid extremely high fares, pulling the mean up.

**Follow-up:** "What transformations can you apply to reduce skewness?"

---

### 5. What is multicollinearity?

**Simple answer:** Multicollinearity is when two or more features are highly correlated with each other, making it hard to tell their individual effects.

**Technical answer:** Multicollinearity occurs when independent variables in a regression model are highly correlated. It causes:
- Unstable coefficient estimates (small data changes cause large swings)
- Inflated standard errors and unreliable p-values
- Difficulty in interpreting feature importance
- Poor generalization in linear models

**Detection methods:**
- Correlation matrix: |r| > 0.8 between features
- VIF (Variance Inflation Factor): VIF > 5 or 10 indicates problematic multicollinearity

**Example:** In Titanic, Fare and Pclass are correlated (r = -0.55), but not extreme. However, if we created FamilySize = SibSp + Parch, it would be highly correlated with its components, causing multicollinearity.

**Follow-up:** "How would you fix multicollinearity in your dataset?"

---

### 6. What tools do you use for EDA?

**Simple answer:** I use Python libraries like Pandas, Matplotlib, Seaborn, and Plotly for statistics and charts.

**Technical answer:** My EDA toolkit includes:
- **Pandas:** `describe()`, `info()`, `value_counts()`, `groupby()`, `corr()`, `isnull()`
- **Matplotlib/Seaborn:** Histograms, boxplots, violin plots, heatmaps, pairplots, count plots
- **Plotly:** Interactive scatter plots, 3D plots, dashboards for stakeholder exploration
- **Scipy:** Statistical tests (z-score, t-test, chi-square)
- **Yellowbrick / Sweetviz / Pandas Profiling:** Automated EDA reports for quick overviews

**Example:** For Titanic, I start with `df.describe()` and `df.info()`, then use Seaborn heatmaps for correlation, pairplots for relationships, and Plotly for interactive exploration of Fare vs Age by Survival.

**Follow-up:** "Have you used any automated EDA tools? What are their pros and cons?"

---

### 7. Can you explain a time when EDA helped you find a problem?

**Simple answer:** Yes — EDA once revealed that a dataset had impossible values (negative ages) and duplicate rows that would have ruined the model.

**Technical answer:** In a previous project analyzing customer churn, EDA revealed:
- **Problem:** The `tenure` column had negative values for ~8% of rows, which was impossible (tenure cannot be negative)
- **Root cause:** A bug in the data pipeline where dates were subtracted in the wrong order during ETL
- **Impact:** Without EDA, these invalid rows would have trained the model on impossible patterns, causing poor generalization
- **Resolution:** Flagged the data engineering team, applied a filter `df[df['tenure'] >= 0]`, and validated the fix with a follow-up EDA check

**Example:** Another time, a boxplot revealed that 15% of `monthly_charges` values were exactly $0, which was unexpected. Investigation showed these were new customers on free trials — a segment the business wanted to analyze separately.

**Follow-up:** "How do you communicate EDA findings to non-technical stakeholders?"

---

### 8. What is the role of visualization in ML?

**Simple answer:** Visualization helps us see patterns, problems, and relationships in data that raw numbers hide.

**Technical answer:** Visualization plays critical roles throughout the ML lifecycle:
- **EDA:** Reveal distributions, outliers, correlations, and class balance
- **Feature engineering:** Guide transformations (e.g., log transform for skewed data)
- **Model evaluation:** Confusion matrices, ROC curves, residual plots, learning curves
- **Interpretability:** SHAP plots, feature importance charts, partial dependence plots
- **Debugging:** Identify data leakage, overfitting, or concept drift
- **Communication:** Present results to stakeholders with clear, intuitive charts

**Example:** A confusion matrix makes it immediately obvious if a model is biased toward one class. A SHAP summary plot explains which features drive individual predictions, building trust with stakeholders.

**Follow-up:** "What chart would you use to show imbalanced classes to a stakeholder?"


## Conclusion

This EDA notebook comprehensively analyzed the Titanic dataset:

1. **Summary statistics** revealed missing values, skewness, and data ranges
2. **Univariate analysis** (histograms, boxplots) showed distributions and outliers
3. **Bivariate analysis** identified strong predictors: Sex, Pclass, Fare
4. **Correlation analysis** detected relationships and potential multicollinearity
5. **Pairplots** visualized multi-feature interactions and class separation
6. **Multivariate analysis** revealed interaction effects (Sex × Pclass)
7. **Anomaly detection** using IQR and z-score methods quantified outliers
8. **Pattern recognition** extracted actionable insights for preprocessing
9. **Interactive visualizations** enabled deeper exploration
10. **Interview Q&A** prepared 8 common EDA questions with simple/technical answers

### Key Takeaways
- Always start with `df.info()` and `df.describe()` to understand structure and quality
- Visualize distributions before deciding on imputation or transformation
- Correlation helps identify redundant features and target relationships
- Outliers are not always errors — investigate before removing
- EDA findings should directly inform preprocessing and modeling decisions
- Visualization is not optional — it is how you communicate data stories

---

## How to Use This Notebook

1. Run cells sequentially from top to bottom
2. Observe each plot and read the interpretation
3. Compare statistical outputs with visual patterns
4. Use the inferences to justify preprocessing choices in Task 1
5. Review interview Q&A before technical discussions
